1. 앞서 작성한 `parser.py`의 함수들을 이용해서 pdf, ppt, excel, word 파일을 parsing해서 준비해 봅시다.

2. `chunk_splitter`를 사용해 chunk로 분할해 봅시다!

3. 그리고 모든 chunk를 합쳐 `total_chunks`라는 이름의 변수를 만들어 봅시다.

In [1]:
from parsing import *

word_docs = parse_word("./data/키키테크_사내규정_행동강령.docx")
pdf_docs = parse_pdf("./data/키키테크_AI솔루션_제품카탈로그.pdf")
excel_docs = parse_excel("./data/키키테크_임직원및프로젝트현황.xlsx")
ppt_docs = parse_pptx("./data/키키테크_회사소개.pptx")

word_chunks = chunk_splitter(word_docs)
pdf_chunks = chunk_splitter(pdf_docs)
ppt_chunks = chunk_splitter(ppt_docs)
excel_chunks = chunk_splitter(excel_docs)

total_chunks = word_chunks + pdf_chunks + ppt_chunks + excel_chunks

In [2]:
print(len(total_chunks))
print(total_chunks[0])

124
page_content='(주)키키테크

임직원 행동강령 및 사내 규정 안내서

Ver. 2025.01 | 인사팀 발행

모든 임직원은 본 안내서를 숙지하고 준수하여야 합니다.

목차

1장. 회사 소개 및 핵심 가치

2장. 임직원 행동강령

3장. 근무 규정

4장. 복리후생 안내

5장. IT 보안 정책

6장. 성희롱·괴롭힘 예방 정책

7장. 비상 연락 및 안전 규정




1장. 회사 소개 및 핵심 가치

1.1 회사 개요

(주)키키테크는 2015년 설립된 AI 솔루션 전문기업으로, 기업용 데이터 분석 플랫폼과 AI 에이전트 개발 서비스를 제공합니다. 현재 서울 강남구 테헤란로 본사를 비롯하여 판교, 부산 지사를 운영하고 있으며, 임직원 수는 약 350명입니다.

회사는 '기술로 연결하는 더 나은 내일'이라는 비전 아래, 고객사의 디지털 전환을 돕는 다양한 AI 솔루션을 연구·개발하고 있습니다.

1.2 핵심 가치 (Core Values)' metadata={'source': './data/키키테크_사내규정_행동강령.docx'}


### 2. 텍스트 벡터로 변환

이제 llm이 이해할 수 있는 숫자 벡터 형태로 텍스트를 변환해 줘야 합니다.

그 방식에는 2가지가 있습니다.

1. Sparse Retriever (BM25) : 두 문서에 공통적으로 등장한 단어가 얼마나 많은 지를 비교합니다.
2. Dense Retriever (DL) : 딥러닝 모델을 이용해 문장의 내제적 의미를 담은 벡터를 서로 비교합니다.

Sparse Retriever에서 가장 많이 사용되는 알고리즘 중 하나인 BM25입니다.

langchain에서 기본적으로 제공해 주기도 합니다.

In [ ]:
from langchain_community.retrievers import BM25Retriever

# BM25 sparse retriever 생성 (rank_bm25 패키지 필요: pip install rank-bm25)
bm25_retriever = BM25Retriever.from_documents(total_chunks)
bm25_retriever.k = 3  # 상위 3개 문서 반환, 가장 유사한 3개 가져오기.

# 테스트
results = bm25_retriever.invoke("키키테크의 주요 수익모델은 뭐야?")
for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(f"출처: {doc.metadata.get('source', '엑셀 데이터')}")
    print(f"내용: {doc.page_content[:200]}")


--- Result 1 ---
출처: ./data/키키테크_AI솔루션_제품카탈로그.pdf
내용: 9. 가격 정책 및 라이선스
키키테크의 모든 제품은 구독형(SaaS)과 영구 라이선스(온프레미스) 두 가지 방식으로
제공됩니다. 아래 가격은 기준 가격이며, 도입 규모, 계약 기간, 번들 구성에 따라 협의
할인이 가능합니다.
TalkBridge Enterprise 가격표
 플랜
 월 구독료
 포함 사용자
 문서 한도
 Starter
 월 200만원
 50명

--- Result 2 ---
출처: ./data/키키테크_사내규정_행동강령.docx
내용: 1.2 핵심 가치 (Core Values)

혁신 (Innovation): 새로운 기술과 아이디어를 두려움 없이 탐구하고 도전합니다.

신뢰 (Trust): 고객, 동료, 파트너와의 관계에서 언제나 투명하고 책임감 있게 행동합니다.

협력 (Collaboration): 다양한 배경과 역량을 가진 구성원들이 함께 더 큰 성과를 만들어냅니다.

성장 (Grow

--- Result 3 ---
출처: ./data/키키테크_AI솔루션_제품카탈로그.pdf
내용: 3. DataSight AI — 지능형 데이터 분석 플랫폼
3.1 제품 개요
DataSight AI는 비전문가도 자연어로 데이터를 분석하고 시각화할 수 있는 지능형 데이터 분석
플랫폼입니다. '지난 분기 지역별 매출 상위 5개 제품을 보여줘'와 같은 자연어 질의만으로
복잡한 SQL 쿼리와 시각화 차트를 자동으로 생성합니다.
3.2 주요 기능
 NL2SQL


Dense Retriever를 만들기 위해선 딥러닝 모델이 필요합니다.

여기에는 텍스트 생성 모델(DS Assistant)이 아니라 텍스트 임베딩에 특화된 언어 모델이 사용됩니다.

In [6]:
# 임베딩 모델 선언
from dotenv import load_dotenv, find_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv(find_dotenv())

base_url="https://mlapi.run/b54ff33e-6d14-42df-93f9-0f1132160ee8/v1"

model_name="openai/text-embedding-3-small"

embedding_model = OpenAIEmbeddings(
    model=model_name,
    openai_api_base=base_url
)

정의된 임베딩 모델을 간단하게 사용해 보겠습니다.

In [7]:
query_example = embedding_model.embed_query("랭체인이란 뭘까요?")

print(query_example[:5])

[-0.0027599334716796875, -0.0160675048828125, -0.00125885009765625, 0.03497314453125, 0.031585693359375]


일반적으로 Dense Retriever를 사용할 때는 빠른 검색을 위하여 FAISS라는 도구를 함꼐 사용합니다.

#### FAISS란?

의미가 비슷한 것끼리 빠르게 찾아주는 도서관 사서와 같은 역할을 합니다.

벡터 형태를 빠르게 찾을 수 있도록 정렬해서 보관(vectorstore)해 준다고 생각하시면 됩니다!

In [10]:
from langchain_community.vectorstores import FAISS


vectorstore = FAISS.from_documents(total_chunks, embedding_model)

벡터는 이렇게 생겼습니다.

In [12]:
import numpy as np

vectors = np.zeros((vectorstore.index.ntotal, vectorstore.index.d), dtype=np.float32)
vectorstore.index.reconstruct_n(0, vectorstore.index.ntotal, vectors)

print(vectors.shape)
print(vectors[0][:10])

(124, 1536)
[-0.01173401  0.01922607 -0.01644897  0.00665283  0.03939819 -0.02055359
 -0.04052734  0.0534668  -0.01507568 -0.01863098]


`vectorstore.similarity_search()`를 통해 빠르게 유사 문서를 찾을 수 있습니다.

In [14]:
results = vectorstore.similarity_search("키키테크의 주요 수익 모델은?", k=3)

for i, doc in enumerate(results, 1):
    print(f"출처: {doc.metadata.get('source')}")
    print(f"내용: {doc.page_content[:200]}")

출처: ./data/키키테크_회사소개.pptx
내용: (주)키키테크
회사 소개
기술로 연결하는 더 나은 내일
2025
2015
설립연도
350+
임직원 수
480억
2024 매출
출처: ./data/키키테크_AI솔루션_제품카탈로그.pdf
내용: 1. 회사 개요 및 솔루션 포트폴리오
1.1 (주)키키테크 소개
(주)키키테크는 2015년 설립된 국내 대표 AI 솔루션 전문기업입니다. 창립 이래 기업의 디지털
전환을 선도하는 혁신적인 AI 기술을 연구·개발해 왔으며, 현재 금융, 제조, 유통, 의료,
법률 등 다양한 산업군의 350여 개 기업 고객을 보유하고 있습니다.
서울 강남구 테헤란로에 본사를 두
출처: ./data/키키테크_AI솔루션_제품카탈로그.pdf
내용: 분석 → 전처리 → 피처 엔지니어링 → 모델 선택(XGBoost, LightGBM, Prophet, LSTM 등 15종)
→ 하이퍼파라미터 최적화 → 앙상블 → 배포의 7단계 파이프라인이 자동으로 실행됩니다.
내부 벤치마크에서 숙련된 데이터 과학자가 수동으로 구축한 모델 대비 92%의 성능을
달성하면서, 개발 시간은 평균 3주에서 4시간으로 단축되었습니다.


생성된 벡터는 저장할 수 있습니다.

In [15]:
vectorstore.save_local("faiss_index")

아래 코드로 다시 불러오는 것도 가능합니다.

In [16]:
vectorstore = FAISS.load_local(
    "faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

### Ensemble Retriever

일반적으로 retriever에는 sparse retriever와 dense retriever를 일정 비율로 섞어서 관련 문서를 찾아옵니다.

`EnsembleRetriever`는 두 retriever의 결과를 **RRF(Reciprocal Rank Fusion)** 알고리즘으로 합산합니다.

| 방식 | 장점 | 단점 |
|------|------|------|
| **Sparse (BM25)** | 고유명사·숫자 등 키워드 정확 매칭 | 의미 유사도 파악 불가 |
| **Dense (FAISS)** | 문맥·의미 기반 유사도 검색 | 정확한 키워드 매칭 약함 |
| **Hybrid (Ensemble)** | 두 방식의 장점 보완 | — |

`weights`로 각 retriever의 기여 비율을 조절할 수 있으며, 합이 1이 되어야 합니다.

In [17]:
from langchain_classic.retrievers import EnsembleRetriever

# FAISS dense retriever
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Sparse(BM25) 40% + Dense(FAISS) 60% 비율로 혼합
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.4, 0.6]
)

# 테스트: 동일한 질문으로 하이브리드 검색 결과 확인
results = ensemble_retriever.invoke("키키테크의 주요 수익모델은 뭐야?")
for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(f"출처: {doc.metadata.get('source', '엑셀 데이터')}")
    print(f"내용: {doc.page_content[:200]}")


--- Result 1 ---
출처: ./data/키키테크_회사소개.pptx
내용: (주)키키테크
회사 소개
기술로 연결하는 더 나은 내일
2025
2015
설립연도
350+
임직원 수
480억
2024 매출

--- Result 2 ---
출처: ./data/키키테크_AI솔루션_제품카탈로그.pdf
내용: 1. 회사 개요 및 솔루션 포트폴리오
1.1 (주)키키테크 소개
(주)키키테크는 2015년 설립된 국내 대표 AI 솔루션 전문기업입니다. 창립 이래 기업의 디지털
전환을 선도하는 혁신적인 AI 기술을 연구·개발해 왔으며, 현재 금융, 제조, 유통, 의료,
법률 등 다양한 산업군의 350여 개 기업 고객을 보유하고 있습니다.
서울 강남구 테헤란로에 본사를 두

--- Result 3 ---
출처: ./data/키키테크_AI솔루션_제품카탈로그.pdf
내용: (주)키키테크
 AI 솔루션 제품 카탈로그 2025
 본 카탈로그는 (주)키키테크가 제공하는 엔터프라이즈 AI 솔루션의 전체 라인업을
 소개합니다.
각 제품의 기술 사양, 적용 사례, 가격 정책을 포함하고 있으며 도입 검토에 활용하실
 수 있습니다.
목 차
1. 회사 개요 및 솔루션 포트폴리오
2. TalkBridge Enterprise — 기업용 AI 챗

--- Result 4 ---
출처: ./data/키키테크_AI솔루션_제품카탈로그.pdf
내용: 9. 가격 정책 및 라이선스
키키테크의 모든 제품은 구독형(SaaS)과 영구 라이선스(온프레미스) 두 가지 방식으로
제공됩니다. 아래 가격은 기준 가격이며, 도입 규모, 계약 기간, 번들 구성에 따라 협의
할인이 가능합니다.
TalkBridge Enterprise 가격표
 플랜
 월 구독료
 포함 사용자
 문서 한도
 Starter
 월 200만원
 50명

--- Result 5 ---
출처: ./data/키키테크_사내규정_행동강령.docx
내용: 1.2 핵심 가치 (Core Values)

혁신 (Innovation): 새로운 기술과 아이디어를 두려움 없

이렇게 만든 retriever를 사용해서 바로 챗봇에 적용해 보도록 하겠습니다!

챗봇에 적용하는 방법은 위에서 만든 retriever를 tool로 제공해 주는 것입니다.

In [18]:
from langchain_core.tools import tool
from pathlib import Path


@tool
def search_documents(query: str) -> str:
    """
    키키테크의 회사 정보, 사내 규정(행동강령, 복리후생, 보안), 제품 카탈로그(TalkBridge, DataSight),
    임직원 및 프로젝트 현황 문서를 검색하여 관련 본문 내용을 반환합니다.
    """
    # 1. 위에서 만든 하이브리드 검색기(ensemble_retriever)로 문서를 찾습니다.
    results = ensemble_retriever.invoke(query)
    
    # 2. 검색된 문서들을 읽기 좋은 형태로 가공하여 하나로 합칩니다.
    formatted_results = []
    for doc in results:
        # 파일 전체 경로(예: data/회사소개.pptx)에서 파일명(회사소개.pptx)만 깨끗하게 추출합니다.
        source_name = Path(doc.metadata.get("source", "unknown")).name
        
        # [출처: 파일명]과 글 본문 내용을 묶어 리스트에 넣습니다.
        formatted_results.append(f"[출처: {source_name}]\n{doc.page_content}")
        
    # 3. 각 결과 사이에 '---' 구분선을 넣어서 하나의 긴 글자로 연결해 반환합니다.
    return "\n\n---\n\n".join(formatted_results)

챗봇을 직접 만들어서 tool로 제공한 뒤에 질문을 직접 해봅시다!